# Training a Small Transformer from Scratch

---

## Learning Objectives

By the end of this project, you will:

- Implement all core transformer components from scratch in PyTorch:
  - Positional encoding
  - Scaled dot-product attention
  - Multi-head attention
  - Transformer encoder block
  - Full transformer encoder classifier
- Prepare text data for transformer input (tokenize, build vocabulary, create sequences)
- Train a transformer with learning rate warmup and early stopping
- Visualize attention weights to understand what the model learns
- Compare your from-scratch transformer with a HuggingFace pretrained model (optional)

## Prerequisites

- **DL100-DL200**: Neural network and MLP fundamentals
- **DL400**: RNN/LSTM (for context on sequence modeling)
- **DL500**: NLP text preprocessing
- **DL600**: Attention and transformer concepts
- Comfortable reading PyTorch `nn.Module` code

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Data Preparation](#2.-Data-Preparation)
3. [Positional Encoding](#3.-Positional-Encoding)
4. [Multi-Head Attention](#4.-Multi-Head-Attention)
5. [Transformer Encoder Block](#5.-Transformer-Encoder-Block)
6. [Full Transformer Encoder Classifier](#6.-Full-Transformer-Encoder-Classifier)
7. [Training with LR Warmup and Early Stopping](#7.-Training)
8. [Evaluation](#8.-Evaluation)
9. [Attention Visualization](#9.-Attention-Visualization)
10. [OPTIONAL: HuggingFace Comparison](#10.-HuggingFace-Comparison)
11. [Conclusions](#11.-Conclusions)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")
from src.utils.seed import set_global_seed
set_global_seed(42)

from src.utils.device import get_device
from src.utils.training import EarlyStopping
from src.utils.text_preprocessing import (
    normalize_text, basic_tokenize, build_vocab, texts_to_sequences
)

import math
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = get_device()
print("Setup complete.")

---

## 2. Data Preparation

We use the same 4-category subset of 20 Newsgroups as Project 02, with preprocessing via our utility module.

In [ ]:
CATEGORIES = [
    "sci.space", "rec.sport.hockey",
    "comp.graphics", "talk.politics.mideast",
]

USE_REAL_DATA = True
try:
    newsgroups_train = fetch_20newsgroups(
        subset="train", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42
    )
    newsgroups_test = fetch_20newsgroups(
        subset="test", categories=CATEGORIES,
        remove=("headers", "footers", "quotes"), random_state=42
    )
    texts_all_train = newsgroups_train.data
    y_all_train = newsgroups_train.target
    texts_test = newsgroups_test.data
    y_test_np = newsgroups_test.target
    CLASS_NAMES = [c.split(".")[-1] for c in CATEGORIES]
    NUM_CLASSES = len(CATEGORIES)
    print(f"20 Newsgroups loaded: {len(texts_all_train)} train, {len(texts_test)} test")
except Exception as e:
    print(f"Failed to load 20 Newsgroups: {e}. Using synthetic data.")
    USE_REAL_DATA = False

In [ ]:
# Fallback: synthetic text data
if not USE_REAL_DATA:
    set_global_seed(42)
    CLASS_NAMES = ["space", "hockey", "graphics", "mideast"]
    NUM_CLASSES = 4
    keyword_pools = {
        0: ["nasa", "orbit", "shuttle", "rocket", "satellite", "launch", "astronaut",
            "space", "moon", "mars", "telescope", "galaxy", "astronomy", "comet"],
        1: ["hockey", "goal", "puck", "rink", "nhl", "team", "player", "score",
            "goalie", "penalty", "period", "assist", "defenseman", "playoff"],
        2: ["graphics", "image", "render", "pixel", "opengl", "texture", "shader",
            "display", "resolution", "algorithm", "polygon", "ray", "3d", "color"],
        3: ["israel", "arab", "peace", "conflict", "government", "political", "war",
            "region", "negotiations", "settlement", "territory", "policy", "united"],
    }
    filler = ["the", "a", "is", "was", "and", "of", "to", "in", "for", "with",
              "on", "that", "this", "it", "from", "has", "have", "been", "are", "were"]

    def generate_doc(label, rng, min_len=30, max_len=80):
        length = rng.randint(min_len, max_len)
        keywords = keyword_pools[label]
        n_kw = max(5, length // 3)
        words = list(rng.choice(keywords, n_kw)) + list(rng.choice(filler, length - n_kw))
        rng.shuffle(words)
        return " ".join(words)

    rng = np.random.RandomState(42)
    texts_all_train, y_all_train = [], []
    texts_test, y_test_np = [], []
    for label in range(NUM_CLASSES):
        for _ in range(400):
            texts_all_train.append(generate_doc(label, rng))
            y_all_train.append(label)
        for _ in range(100):
            texts_test.append(generate_doc(label, rng))
            y_test_np.append(label)
    y_all_train = np.array(y_all_train)
    y_test_np = np.array(y_test_np)
    print(f"Synthetic data: {len(texts_all_train)} train, {len(texts_test)} test")

In [ ]:
# Preprocess, split, build vocab, convert to sequences
texts_train_raw, texts_val_raw, y_train_np, y_val_np = train_test_split(
    texts_all_train, y_all_train, test_size=0.15, random_state=42, stratify=y_all_train
)

texts_train_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_train_raw]
texts_val_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_val_raw]
texts_test_clean = [normalize_text(t, remove_punctuation=True, remove_numbers=True) for t in texts_test]

MAX_VOCAB_SIZE = 10000
MAX_SEQ_LEN = 128

vocab = build_vocab(texts_train_clean, max_vocab_size=MAX_VOCAB_SIZE, min_freq=2)
VOCAB_SIZE = len(vocab)
PAD_IDX = vocab["<PAD>"]

X_train_seq = texts_to_sequences(texts_train_clean, vocab, max_len=MAX_SEQ_LEN)
X_val_seq = texts_to_sequences(texts_val_clean, vocab, max_len=MAX_SEQ_LEN)
X_test_seq = texts_to_sequences(texts_test_clean, vocab, max_len=MAX_SEQ_LEN)

X_train_t = torch.tensor(X_train_seq, dtype=torch.long)
X_val_t = torch.tensor(X_val_seq, dtype=torch.long)
X_test_t = torch.tensor(X_test_seq, dtype=torch.long)
y_train_t = torch.tensor(y_train_np, dtype=torch.long)
y_val_t = torch.tensor(y_val_np, dtype=torch.long)
y_test_t = torch.tensor(y_test_np, dtype=torch.long)

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

print(f"Vocab size: {VOCAB_SIZE} | Max seq len: {MAX_SEQ_LEN}")
print(f"Train: {len(X_train_t)} | Val: {len(X_val_t)} | Test: {len(X_test_t)}")

---

## 3. Positional Encoding

Unlike RNNs, transformers process all tokens in parallel and have no inherent notion of position. We add **positional encoding** so the model knows where each token appears in the sequence.

The standard sinusoidal encoding from "Attention Is All You Need":

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding.

    Adds position information to token embeddings so the transformer
    can distinguish token order.
    """

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create positional encoding matrix: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices

        # Register as buffer (not a parameter, but saved with model)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """Add positional encoding to input embeddings.

        Args:
            x: (batch_size, seq_len, d_model)
        Returns:
            (batch_size, seq_len, d_model) with positional info added
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Visualize positional encoding
pe_module = PositionalEncoding(d_model=64, max_len=128)
pe_values = pe_module.pe[0].numpy()  # (128, 64)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pe_values, aspect="auto", cmap="RdBu_r", interpolation="nearest")
ax.set_xlabel("Embedding Dimension")
ax.set_ylabel("Position")
ax.set_title("Sinusoidal Positional Encoding")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

---

## 4. Multi-Head Attention

The core mechanism of the transformer. Multi-head attention allows the model to attend to information from different representation subspaces at different positions.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention mechanism.

    Splits Q, K, V into multiple heads, computes scaled dot-product
    attention independently, then concatenates and projects.
    """

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.attn_weights = None  # stored for visualization

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Compute scaled dot-product attention.

        Args:
            Q: (batch, heads, seq_len, d_k)
            K: (batch, heads, seq_len, d_k)
            V: (batch, heads, seq_len, d_k)
            mask: optional (batch, 1, 1, seq_len) padding mask

        Returns:
            output: (batch, heads, seq_len, d_k)
            attn_weights: (batch, heads, seq_len, seq_len)
        """
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, V)
        return output, attn_weights

    def forward(self, x, mask=None):
        """Forward pass for self-attention.

        Args:
            x: (batch, seq_len, d_model)
            mask: optional padding mask

        Returns:
            output: (batch, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.shape

        # Linear projections
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Split into heads: (batch, seq_len, d_model) -> (batch, heads, seq_len, d_k)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        self.attn_weights = attn_weights.detach()  # store for visualization

        # Concatenate heads: (batch, heads, seq_len, d_k) -> (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model
        )

        # Final linear projection
        output = self.W_o(attn_output)
        return output


# Quick test
mha = MultiHeadAttention(d_model=64, num_heads=4)
test_input = torch.randn(2, 10, 64)  # (batch=2, seq_len=10, d_model=64)
test_output = mha(test_input)
print(f"Input shape:  {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Attention weights shape: {mha.attn_weights.shape}")

---

## 5. Transformer Encoder Block

Each encoder block consists of:
1. Multi-head self-attention + residual connection + layer norm
2. Position-wise feed-forward network + residual connection + layer norm

```
x -> MultiHeadAttention -> Add & LayerNorm -> FFN -> Add & LayerNorm -> output
     |______________________|                  |_______|  
         (residual)                            (residual)
```

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Single Transformer Encoder block.

    Contains multi-head self-attention followed by a position-wise
    feed-forward network, each with residual connections and layer norm.
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # Multi-head attention sublayer
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        # Feed-forward sublayer
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """Forward pass.

        Args:
            x: (batch, seq_len, d_model)
            mask: optional padding mask
        Returns:
            (batch, seq_len, d_model)
        """
        # Self-attention with residual + norm
        attn_out = self.attention(x, mask)
        x = self.norm1(x + self.dropout1(attn_out))

        # FFN with residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        return x


# Test
enc_block = TransformerEncoderBlock(d_model=64, num_heads=4, d_ff=256)
test_out = enc_block(test_input)
print(f"Encoder block: {test_input.shape} -> {test_out.shape}")

---

## 6. Full Transformer Encoder Classifier

The complete model stacks:
1. Token embedding + positional encoding
2. N transformer encoder blocks
3. Global average pooling over the sequence
4. Classification head

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer Encoder for text classification.

    Architecture:
        Embedding -> PositionalEncoding -> N x EncoderBlock
        -> Global Avg Pool -> Linear -> num_classes
    """

    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 num_classes, max_len=512, dropout=0.1, pad_idx=0):
        super().__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx

        # Embedding + positional encoding
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)

        # Transformer encoder blocks
        self.encoder_blocks = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        """Initialize weights with Xavier uniform."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _create_padding_mask(self, x):
        """Create mask where 1 = real token, 0 = padding."""
        # x: (batch, seq_len)
        mask = (x != self.pad_idx).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
        return mask.float()

    def forward(self, x):
        """Forward pass.

        Args:
            x: (batch, seq_len) integer token IDs
        Returns:
            logits: (batch, num_classes)
        """
        # Create padding mask
        mask = self._create_padding_mask(x)

        # Embedding + positional encoding (scale by sqrt(d_model))
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)

        # Pass through encoder blocks
        for block in self.encoder_blocks:
            x = block(x, mask)

        # Global average pooling (ignoring padding)
        # mask: (batch, 1, 1, seq_len) -> (batch, seq_len, 1)
        pad_mask = mask.squeeze(1).squeeze(1).unsqueeze(-1)  # (batch, seq_len, 1)
        x = (x * pad_mask).sum(dim=1) / pad_mask.sum(dim=1).clamp(min=1)

        # Classification
        logits = self.classifier(x)
        return logits


# Hyperparameters for our small transformer
D_MODEL = 128
NUM_HEADS = 4
D_FF = 256
NUM_LAYERS = 3
DROPOUT = 0.2

set_global_seed(42)
transformer_model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
    max_len=MAX_SEQ_LEN, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(device)

n_params = sum(p.numel() for p in transformer_model.parameters())
print(transformer_model)
print(f"\nTotal parameters: {n_params:,}")

---

## 7. Training

We train with:
- **Learning rate warmup**: linearly increase LR for the first few epochs, then decay
- **Early stopping**: halt when validation loss stops improving
- **Gradient clipping**: prevent exploding gradients

In [ ]:
class WarmupScheduler:
    """Learning rate scheduler with linear warmup followed by cosine decay."""

    def __init__(self, optimizer, d_model, warmup_steps):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.current_step = 0

    def step(self):
        self.current_step += 1
        lr = self._compute_lr()
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr

    def _compute_lr(self):
        step = self.current_step
        # "Attention Is All You Need" warmup schedule
        lr = (self.d_model ** -0.5) * min(
            step ** -0.5,
            step * (self.warmup_steps ** -1.5)
        )
        return lr


def train_transformer(model, train_loader, val_loader, epochs, warmup_steps,
                      patience, d_model, device):
    """Train the transformer with warmup and early stopping."""
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)
    scheduler = WarmupScheduler(optimizer, d_model, warmup_steps)
    loss_fn = nn.CrossEntropyLoss()
    early_stop = EarlyStopping(patience=patience)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            running_loss += loss.item() * X_batch.size(0)
            correct += (output.argmax(1) == y_batch).sum().item()
            total += y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total
        current_lr = optimizer.param_groups[0]["lr"]

        # Validate
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = loss_fn(output, y_batch)
                running_loss += loss.item() * X_batch.size(0)
                correct += (output.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss = running_loss / total
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train: {train_loss:.4f}/{train_acc:.4f} | "
                  f"Val: {val_loss:.4f}/{val_acc:.4f} | "
                  f"LR: {current_lr:.2e}")

        if early_stop(val_loss, model):
            print(f"Early stopping at epoch {epoch}")
            early_stop.load_best(model)
            break

    elapsed = time.time() - start_time
    history["training_time"] = elapsed
    print(f"\nTraining completed in {elapsed:.1f}s")
    return history

In [ ]:
set_global_seed(42)
transformer_model = TransformerClassifier(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=NUM_HEADS,
    d_ff=D_FF, num_layers=NUM_LAYERS, num_classes=NUM_CLASSES,
    max_len=MAX_SEQ_LEN, dropout=DROPOUT, pad_idx=PAD_IDX,
).to(device)

EPOCHS = 40
WARMUP_STEPS = 500
PATIENCE = 7

print("=" * 60)
print("Training Transformer Encoder Classifier")
print("=" * 60)

history = train_transformer(
    transformer_model, train_loader, val_loader,
    epochs=EPOCHS, warmup_steps=WARMUP_STEPS,
    patience=PATIENCE, d_model=D_MODEL, device=device
)

In [ ]:
# Plot training curves and learning rate schedule
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train", marker="o", markersize=3)
axes[0].plot(ep, history["val_loss"], label="Val", marker="s", markersize=3)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curves")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history["train_acc"], label="Train", marker="o", markersize=3)
axes[1].plot(ep, history["val_acc"], label="Val", marker="s", markersize=3)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy Curves")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep, history["lr"], color="tab:green", marker="o", markersize=3)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule (Warmup)")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8. Evaluation

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    """Collect all predictions and labels."""
    model.eval()
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        output = model(X_batch)
        all_preds.extend(output.argmax(1).cpu().numpy())
        all_labels.extend(y_batch.numpy())
    return np.array(all_labels), np.array(all_preds)


y_true, y_pred = get_predictions(transformer_model, test_loader, device)
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average="weighted")

print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 (weighted): {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title(f"Transformer Classifier (Acc: {acc:.3f})")
plt.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.show()

---

## 9. Attention Visualization

One of the key advantages of transformers is that we can **visualize attention weights** to see which tokens the model focuses on when making predictions.

In [ ]:
def visualize_attention(model, text, vocab, max_len, device, layer_idx=-1, head_idx=0):
    """Visualize attention weights for a single text input.

    Args:
        model: Trained TransformerClassifier
        text: Raw text string
        vocab: Token-to-index mapping
        max_len: Max sequence length
        device: torch device
        layer_idx: Which encoder layer to visualize (-1 = last)
        head_idx: Which attention head to visualize
    """
    model.eval()

    # Preprocess
    clean_text = normalize_text(text, remove_punctuation=True, remove_numbers=True)
    tokens = basic_tokenize(clean_text)
    seq = texts_to_sequences([clean_text], vocab, max_len=max_len)[0]
    input_tensor = torch.tensor([seq], dtype=torch.long).to(device)

    # Forward pass
    with torch.no_grad():
        output = model(input_tensor)
        pred = output.argmax(1).item()

    # Get attention weights from the specified layer
    encoder_block = model.encoder_blocks[layer_idx]
    attn_weights = encoder_block.attention.attn_weights  # (1, heads, seq, seq)

    # Select the specified head
    attn = attn_weights[0, head_idx].cpu().numpy()  # (seq, seq)

    # Only show the first n_tokens (not padding)
    n_tokens = min(len(tokens), max_len, 30)  # cap at 30 for readability
    display_tokens = tokens[:n_tokens]
    attn_display = attn[:n_tokens, :n_tokens]

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(attn_display, cmap="viridis", aspect="auto")
    ax.set_xticks(range(n_tokens))
    ax.set_xticklabels(display_tokens, rotation=90, fontsize=8)
    ax.set_yticks(range(n_tokens))
    ax.set_yticklabels(display_tokens, fontsize=8)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.set_title(f"Attention Weights (Layer {layer_idx}, Head {head_idx}) | "
                 f"Pred: {CLASS_NAMES[pred]}")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

    return display_tokens, attn_display, pred

In [ ]:
# Visualize attention for sample documents from each class
sample_texts = []
for c in range(NUM_CLASSES):
    for i, t in enumerate(texts_test):
        if y_test_np[i] == c:
            sample_texts.append(t)
            break

for i, text in enumerate(sample_texts):
    print(f"\nClass: {CLASS_NAMES[i]}")
    print(f"Text preview: {text[:100]}...")
    tokens, attn, pred = visualize_attention(
        transformer_model, text, vocab, MAX_SEQ_LEN, device,
        layer_idx=-1, head_idx=0
    )

In [ ]:
# Compare attention across all heads for one example
sample_text = sample_texts[0]
clean = normalize_text(sample_text, remove_punctuation=True, remove_numbers=True)
tokens = basic_tokenize(clean)[:20]

# Forward pass to populate attention weights
seq = texts_to_sequences([clean], vocab, max_len=MAX_SEQ_LEN)[0]
input_t = torch.tensor([seq], dtype=torch.long).to(device)
with torch.no_grad():
    _ = transformer_model(input_t)

# Plot all heads from the last layer
last_block = transformer_model.encoder_blocks[-1]
attn_all_heads = last_block.attention.attn_weights[0].cpu().numpy()  # (heads, seq, seq)

n_display = min(len(tokens), 20)
fig, axes = plt.subplots(1, NUM_HEADS, figsize=(5 * NUM_HEADS, 5))
if NUM_HEADS == 1:
    axes = [axes]

for h, ax in enumerate(axes):
    ax.imshow(attn_all_heads[h, :n_display, :n_display], cmap="viridis", aspect="auto")
    ax.set_title(f"Head {h}")
    ax.set_xticks(range(n_display))
    ax.set_xticklabels(tokens[:n_display], rotation=90, fontsize=7)
    ax.set_yticks(range(n_display))
    ax.set_yticklabels(tokens[:n_display], fontsize=7)

plt.suptitle("Attention Weights Across All Heads (Last Layer)", fontsize=13)
plt.tight_layout()
plt.show()

---

## 10. OPTIONAL: HuggingFace Comparison

If HuggingFace `transformers` is installed, we compare our from-scratch model against a pretrained `DistilBERT` fine-tuned on the same data.

This section is wrapped in try/except so the notebook runs even without HuggingFace installed.

In [ ]:
HF_AVAILABLE = False
try:
    from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
    from transformers import Trainer, TrainingArguments
    HF_AVAILABLE = True
    print("HuggingFace transformers is available.")
except ImportError:
    print("HuggingFace transformers not installed. Skipping comparison.")
    print("Install with: pip install transformers")

In [ ]:
if HF_AVAILABLE:
    try:
        # Tokenize with DistilBERT tokenizer
        hf_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

        def tokenize_texts(texts, max_length=128):
            return hf_tokenizer(
                texts, padding="max_length", truncation=True,
                max_length=max_length, return_tensors="pt"
            )

        train_enc = tokenize_texts(list(texts_train_raw))
        test_enc = tokenize_texts(list(texts_test))

        class HFDataset(torch.utils.data.Dataset):
            def __init__(self, encodings, labels):
                self.encodings = encodings
                self.labels = labels
            def __len__(self):
                return len(self.labels)
            def __getitem__(self, idx):
                item = {k: v[idx] for k, v in self.encodings.items()}
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
                return item

        hf_train_ds = HFDataset(train_enc, y_train_np.tolist())
        hf_test_ds = HFDataset(test_enc, y_test_np.tolist())

        # Load pretrained DistilBERT and fine-tune
        hf_model = DistilBertForSequenceClassification.from_pretrained(
            "distilbert-base-uncased", num_labels=NUM_CLASSES
        )

        training_args = TrainingArguments(
            output_dir="../data/hf_trainer_output",
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            warmup_steps=100,
            logging_steps=50,
            eval_strategy="epoch",
            save_strategy="no",
            report_to="none",
        )

        trainer = Trainer(
            model=hf_model,
            args=training_args,
            train_dataset=hf_train_ds,
            eval_dataset=hf_test_ds,
        )

        print("Fine-tuning DistilBERT...")
        trainer.train()

        # Evaluate
        hf_preds = trainer.predict(hf_test_ds)
        hf_pred_labels = np.argmax(hf_preds.predictions, axis=1)
        hf_acc = accuracy_score(y_test_np, hf_pred_labels)
        hf_f1 = f1_score(y_test_np, hf_pred_labels, average="weighted")

        print(f"\nDistilBERT Test Accuracy: {hf_acc:.4f}")
        print(f"DistilBERT Test F1: {hf_f1:.4f}")
        print(f"\nOur Transformer Test Accuracy: {acc:.4f}")
        print(f"Our Transformer Test F1: {f1:.4f}")

        print("\nComparison:")
        print(f"  DistilBERT params: ~66M (pretrained on massive corpus)")
        print(f"  Our Transformer params: {n_params:,} (trained from scratch)")

    except Exception as e:
        print(f"HuggingFace comparison failed: {e}")
        print("This is expected if model download fails or GPU memory is insufficient.")
else:
    print("Skipping HuggingFace comparison (not installed).")

---

## 11. Conclusions

### What We Built

We implemented a complete transformer encoder classifier from scratch, including:
- Sinusoidal positional encoding
- Multi-head scaled dot-product attention
- Transformer encoder blocks with residual connections and layer normalization
- A classification head with global average pooling

### Key Insights

| Aspect | Our From-Scratch Transformer | Pretrained (DistilBERT) |
|--------|------------------------------|------------------------|
| **Parameters** | ~100K-500K | ~66M |
| **Training data needed** | All task-specific | Pretrained on billions of tokens |
| **Training time** | Minutes | Minutes (fine-tuning only) |
| **Performance** | Good for small tasks | State-of-the-art |
| **Understanding** | Full control, deep understanding | Black-box but powerful |

### Takeaways

- Building from scratch gives **deep understanding** of the architecture
- **Learning rate warmup** is critical for transformer training stability
- **Attention visualization** reveals what the model focuses on
- For production use, **pretrained transformers** (BERT, DistilBERT) almost always outperform from-scratch models due to transfer learning from massive pretraining corpora
- The from-scratch approach is valuable when you need: a very small model, full control over the architecture, or educational understanding

### Next Steps

- Experiment with more encoder layers and attention heads
- Try learned positional encodings instead of sinusoidal
- Add a decoder for sequence-to-sequence tasks
- Explore different pooling strategies (CLS token, max pooling)
- Pre-train with masked language modeling before fine-tuning